In [1]:
import numpy as np
import numpyro 
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
import stan

import jax
import jax.numpy as jnp
import scipy
from scipy import stats

import time
import copy
import itertools

import multiprocessing as mp
from tqdm import tqdm
from joblib import Parallel, delayed

In [2]:
samples = stats.invwishart.rvs(10, np.array([[5, 0], [0, 1]]), size=10000000)
samples = samples.reshape(-1, 4)

In [3]:
cov = np.cov(samples.T)

In [4]:
cov

array([[2.04258173e-01, 4.48381411e-06, 4.48381411e-06, 5.08062770e-03],
       [4.48381411e-06, 1.78145208e-02, 1.78145208e-02, 1.28313798e-05],
       [4.48381411e-06, 1.78145208e-02, 1.78145208e-02, 1.28313798e-05],
       [5.08062770e-03, 1.28313798e-05, 1.28313798e-05, 8.12809764e-03]])

In [2]:
def sim_data(args):
    # Set seed
#     np.random.seed(args['seed'])
    
    # Simulate true beta
    hist_lk_list = [[[]] for n in range(args['N'])]
    diff_hist_lk_list = [[[]] for n in range(args['N'])]
    
    m_counter = np.zeros(args['N']).astype(int)
    k_counter = np.ones(args['N']).astype(int)

    # Sample parameter for each site
    if args['d']== 1:
        mu1_true = np.random.normal(loc=args['mu_true'], scale=args['Sigma_true'], size=args['N'])
    else:
        mu1_true = np.random.multivariate_normal(mean=args['mu_true'].flatten(), cov=args['Sigma_true'], size=args['N'])

    for t in range(args['T']):
        for n in range(args['N']):
            # Reassign the variable
            m = m_counter[n]
            k = k_counter[n]
            beta = mu1_true[n]

            # Compute inner product of beta
            if args['d'] > 1:
                beta_term = 0
                for degree in range(args['d']):
                    beta_term += beta[degree] * ((k)**(degree)) * (args['delta']**(degree+1))
            else:
                beta_term = beta * args['delta']

            # Accumulate lk
            lkm1 = args['l_0'] if k == 1 else hist_lk_list[n][m][-1][1]
            lk = lkm1 + beta_term + np.random.randn(1).item() * args['sigma0'].item()
            diff_lk = lk - lkm1
            
            # Append to the list
            hist_lk_list[n][m].append((k, lk.item()))
            diff_hist_lk_list[n][m].append((k, diff_lk.item()))

            if lk > args['r_limit'] and t != args['T']-1:
                m_counter[n] += 1
                k_counter[n] = 0
            
                hist_lk_list[n].append([])
                diff_hist_lk_list[n].append([])
                
        # Increment k
        k_counter += 1
        
    return hist_lk_list, diff_hist_lk_list, mu1_true

In [94]:
args_linear = {
        # Hierarchical params
        'delta': 1, 
        'mu_mu': np.array([5]), 'Sigma_mu': np.array([5]), # Prior of mu
        'alpha_Sigma': np.array([6]) , 'beta_Sigma': np.array([5]), # Prior of Sigma
        'sigma0': np.array([1]), 'l_0': np.array([0]), 
        'mu_true': np.array([1]), 'Sigma_true': np.array([0.01]),
        # Simulation params
        'N': 10, 'M': 20, 'K': 80, 'S': 20, 'r_limit': 10, 'd': 1, 'T': 150, 'seed': 1234,
        # Sampling parameters
        'gibbs_T': 5000,  'warm_up': 2000, 'space': 1, 'n_chains': 4
    }
args_quadratic = {
        # Hierarchical params
        'delta': 1, 
        'mu_mu': np.array([[5], [1]]), 'Sigma_mu': np.array([[5, 0], [0, 5]]), # Prior of mu
        'nu_Sigma': np.array([6]) , 'Sigma_Sigma': np.array([[5, 0], [0, 1]]), # Prior of Sigma
        'sigma0': np.array([0.5]), 'l_0': np.array([0]), 
        'mu_true': np.array([[1], [0.3]]), 'Sigma_true': np.array([[0.01, 0], [0, 0.001]]),
        # Simulation params
        'N': 10, 'M': 20, 'K': 80, 'S': 20, 'r_limit': 40, 'd': 2, 'T': 150, 'seed': 1234, 'sim_num':10,
        # Sampling parameters
        'gibbs_T': 2000, 'warm_up': 1000, 'space': 2
    }
# curr_args = copy.deepcopy(args_linear)
# hist_lk, hist_diff_lk, mu_true = sim_data(curr_args, noise_sigma=2)
# hist_diff_lk_flat = []
# for n in range(curr_args['N']):
#     lk_list = list(itertools.chain.from_iterable(hist_diff_lk[n]))
#     hist_diff_lk_flat.append(lk_list)

# ep
# start_time = time.time()
# beta_mean, beta_var = centralized_gibbs_sampler(curr_args, hist_diff_lk_flat)
# print("--- %s seconds ---" % (time.time() - start_time))

In [95]:
def run_sim_linear(args, sim_round):
    # Print
    print(f"Running sim#{sim_round}")
    
    # Set seed
    np.random.seed(args['seed'])
    d = args['d']
    
    # Intialize empty list
    collab_mean = np.zeros((args['N'], args['T'])); collab_std = np.zeros((args['N'], args['T']))
    collab_lap_1_mean = np.zeros((args['N'], args['T'])); collab_lap_1_std = np.zeros((args['N'], args['T']))
    collab_lap_2_mean = np.zeros((args['N'], args['T'])); collab_lap_2_std = np.zeros((args['N'], args['T']))
    collab_ep_mean = np.zeros((args['N'], args['T'])); collab_ep_std = np.zeros((args['N'], args['T']))
    
    # Sim data
    hist_lk, hist_diff_lk, mu_true = sim_data(args)
    priors = {
        'mu_mu': args['mu_mu'],
        'Sigma_mu': args['Sigma_mu'],
        'alpha_Sigma': args['alpha_Sigma'],
        'beta_Sigma': args['beta_Sigma']
    }
    # Run sim
    for t in range(1, args['T']):
        hist_diff_lk_flat = []
        for n in range(args['N']):
            lk_list = list(itertools.chain.from_iterable(hist_diff_lk[n]))
            hist_diff_lk_flat.append(lk_list[:t])

        # Run experiment for centralized model with no noise
        beta_mean, beta_std = centralized_update(args, priors, hist_diff_lk_flat, noise_sigma=0)
#         beta_mean_iso, beta_std_iso = isolated_update(args, hist_diff_lk_flat)
#         r, Q, r_list_new, Q_list_new = update_EP_posterior_linear(r_list, Q_list, r, Q, hist_diff_lk_flat, args)
#         beta_mean_ep, beta_std_ep = 
#         beta_mean_lap_1, beta_std_lap_1 = centralized_update(args, hist_diff_lk_flat, noise_sigma=1)
#         beta_mean_lap_2, beta_std_lap_2 = centralized_update(args, hist_diff_lk_flat, noise_sigma=4)

#         print(f"(Collab) t={t} | mean={beta_mean.flatten()}")
#         print(f"(Isolated) t={t} | mean={beta_mean_iso.flatten()}")
#         print(f"Ground truth | {mu_true}")
#         print('-'*20)

#         # Centralized model with noise sigma = 1
        collab_mean[:, t] = np.abs(beta_mean - mu_true); collab_std[:, t] = beta_std
        print(f"t = {t} | collab = {beta_mean}")
        print(f"Truth = {mu_true}")
        print('-'*30)
#         collab_lap_1_mean[:, t] = np.abs(beta_mean_lap_1 - mu_true); collab_lap_1_std[:, t] = beta_std_lap_1
#         collab_lap_2_mean[:, t] = np.abs(beta_mean_lap_2 - mu_true); collab_lap_2_std[:, t] = beta_std_lap_2
        
    return collab_mean

In [104]:
def get_mu_sample(args, mu, Sigma, betas, priors):
    precision = 1 / priors['Sigma_mu']**2 + args['N'] / Sigma**2
    linear_shift = priors['mu_mu'] / priors['Sigma_mu']**2 + np.sum(betas) / Sigma**2
    
    mean = linear_shift / precision
    var = 1 / precision
    
    mu_sample = stats.norm.rvs(loc=mean, scale=np.sqrt(var))
    return mu_sample

def get_Sigma_sample(args, mu, Sigma, betas, priors):
    alpha_param = priors['alpha_Sigma'] + args['N'] / 2
    beta_param = priors['beta_Sigma'] + np.sum(betas - mu)**2 / 2
    
    Sigma_squared_sample = stats.invgamma.rvs(a=alpha_param, scale=beta_param)
    return np.sqrt(Sigma_squared_sample)

def get_betas_sample(args, mu, Sigma, betas, diff_lk_array, priors):
    # Initialize empty betas
    new_betas = np.zeros(args['N'])
    
    # Loop over every site
    for n in range(args['N']):
        k_list = diff_lk_array[n, :, 0]
        diff_lk_list = diff_lk_array[n, :, 1]
        num_data = len(diff_lk_list)
        
        precision = num_data * args['delta']**2 / args['sigma0']**2 + 1 / Sigma**2
        linear_shift = np.sum(diff_lk_list) * args['delta'] / args['sigma0']**2 + mu / Sigma**2
        
        mean = linear_shift / precision
        var = 1 / precision
        
        new_betas[n] = stats.norm.rvs(loc=mean, scale=np.sqrt(var))
        
    return new_betas

def centralized_update(args, priors, diff_lk_input, noise_sigma=0):
    diff_lk_array = np.stack(diff_lk_input)
    
    # Initialize memories
    mu_hist = np.zeros((args['n_chains'], args['gibbs_T']))
    Sigma_hist = np.zeros((args['n_chains'], args['gibbs_T']))
    betas_hist = np.zeros((args['N'], args['n_chains'], args['gibbs_T']))

    # Randomly sample the starting point according to the prior
    mu = stats.norm.rvs(loc=priors['mu_mu'], scale=priors['Sigma_mu'])
    Sigma = stats.invgamma.rvs(a=priors['alpha_Sigma'], scale=priors['beta_Sigma'])
    betas = stats.norm.rvs(loc=mu, scale=Sigma, size=args['N'])
    
    # Run Gibbs
    for chain in range(args['n_chains']):
        for t in range(args['gibbs_T']):      
            # Get a sample from mu
            new_mu = get_mu_sample(args, mu, Sigma, betas, priors)

            # Get a sample from Sigma
            new_Sigma = get_Sigma_sample(args, new_mu, Sigma, betas, priors)

            # Get samples for betas
            new_betas = get_betas_sample(args, new_mu, new_Sigma, betas, diff_lk_array, priors)
            
            # Update params
            mu = new_mu; Sigma = new_Sigma; betas = new_betas
            
            # Append history
            mu_hist[chain, t] = mu
            Sigma_hist[chain, t] = Sigma
            betas_hist[:, chain, t] = betas
    
    # Take only the samples after the warm-up period
    mu_hist = mu_hist[:, args['warm_up']:]
    Sigma_hist = Sigma_hist[:, args['warm_up']:]
    betas_hist = betas_hist[:, :, args['warm_up']:]
    
    # Collapse the array
    mu_hist = mu_hist.flatten()
    Sigma_hist = Sigma_hist.flatten()
    betas_hist = betas_hist.reshape(args['N'], -1)
    
    mean_betas = np.mean(betas_hist, axis=-1)
    std_betas = np.std(betas_hist, axis=-1)
    print(betas_hist.shape)
    
    return mean_betas, std_betas

def isolated_update(args, diff_lk_input):
    d = args['d']
    mu0_mean = np.zeros((args['N'], args['d']))
    mu0_std = np.zeros((args['N'], args['d']))
    
    for n in range(args['N']):
        Sigma_inv = np.linalg.inv(args['Sigma_mu']) if d>1 else np.reciprocal(args['Sigma_mu'])
        
        lk_list_n = np.stack(diff_lk_input[n])
        tt_T = np.zeros((len(diff_lk_input[n]), d))
        for idx in range(d):
            tt_T[:, idx] = (lk_list_n[:, 0] ** idx) * (args['delta']**(idx+1))

        new_diff_lk = lk_list_n[:, 1].reshape(-1, 1)
        A = Sigma_inv + tt_T.T @ tt_T / (args['sigma0']**2)
        b = np.dot(Sigma_inv, args['mu_mu']) + (np.sum(np.multiply(new_diff_lk, tt_T), axis=0) / (args['sigma0']**2)).reshape(-1, 1)

        mu0_mean[n] = np.linalg.solve(A, b).flatten()
        mu0_cov = np.linalg.inv(A)
        
        mu0_std[n] = np.sqrt(np.diag(mu0_cov))

    return mu0_mean, mu0_std

In [105]:
collab_mean = run_sim_linear(args_linear, 0)

Running sim#0
(10, 12000)
t = 1 | collab = [1.5754866  1.51452321 1.50371691 0.19472174 0.93605826 1.08087023
 1.25202775 1.20267895 1.64823727 0.40885684]
Truth = [1.00471435 0.98809024 1.01432707 0.99687348 0.99279411 1.00887163
 1.00859588 0.99363476 1.00015696 0.97757315]
------------------------------
(10, 12000)
t = 2 | collab = [1.32496197 1.13588174 1.39973463 0.60458654 1.33145078 0.91119009
 1.36197659 0.58380126 1.38087727 0.88317029]
Truth = [1.00471435 0.98809024 1.01432707 0.99687348 0.99279411 1.00887163
 1.00859588 0.99363476 1.00015696 0.97757315]
------------------------------
(10, 12000)
t = 3 | collab = [1.21174734 1.22629354 1.58502905 0.98941201 1.48924712 0.9465326
 1.36790315 0.64842224 1.53558256 1.50214211]
Truth = [1.00471435 0.98809024 1.01432707 0.99687348 0.99279411 1.00887163
 1.00859588 0.99363476 1.00015696 0.97757315]
------------------------------
(10, 12000)
t = 4 | collab = [1.15809874 1.04810391 1.44712215 0.57180042 1.4203752  0.77099097
 1.244142

(10, 12000)
t = 29 | collab = [1.14917565 0.68035444 0.9535145  1.00522885 1.21770837 0.88882397
 1.05357341 1.15723879 1.08335645 1.3263964 ]
Truth = [1.00471435 0.98809024 1.01432707 0.99687348 0.99279411 1.00887163
 1.00859588 0.99363476 1.00015696 0.97757315]
------------------------------
(10, 12000)
t = 30 | collab = [1.15914262 0.71366055 0.94700181 1.08679084 1.15072545 0.89019128
 1.09308383 1.1683628  1.08792163 1.26152374]
Truth = [1.00471435 0.98809024 1.01432707 0.99687348 0.99279411 1.00887163
 1.00859588 0.99363476 1.00015696 0.97757315]
------------------------------
(10, 12000)
t = 31 | collab = [1.16270524 0.70369301 0.95289055 1.11840058 1.14603402 0.8839026
 1.11899261 1.16274357 1.05936668 1.22084357]
Truth = [1.00471435 0.98809024 1.01432707 0.99687348 0.99279411 1.00887163
 1.00859588 0.99363476 1.00015696 0.97757315]
------------------------------
(10, 12000)
t = 32 | collab = [1.18845859 0.74000246 0.98032056 1.11832784 1.12074703 0.87662253
 1.17593303 1.16170

(10, 12000)
t = 57 | collab = [1.12777186 0.91341223 1.11049958 1.01264844 0.87341583 0.9190047
 1.17767693 1.13178192 1.11014862 0.93487507]
Truth = [1.00471435 0.98809024 1.01432707 0.99687348 0.99279411 1.00887163
 1.00859588 0.99363476 1.00015696 0.97757315]
------------------------------
(10, 12000)
t = 58 | collab = [1.10313763 0.91123664 1.12456931 1.0115301  0.86286624 0.92556223
 1.15440486 1.13211549 1.0952181  0.95883307]
Truth = [1.00471435 0.98809024 1.01432707 0.99687348 0.99279411 1.00887163
 1.00859588 0.99363476 1.00015696 0.97757315]
------------------------------
(10, 12000)
t = 59 | collab = [1.1120493  0.90995793 1.13252602 0.99231949 0.85714357 0.92608384
 1.13431048 1.11215566 1.10959899 0.98908221]
Truth = [1.00471435 0.98809024 1.01432707 0.99687348 0.99279411 1.00887163
 1.00859588 0.99363476 1.00015696 0.97757315]
------------------------------
(10, 12000)
t = 60 | collab = [1.11205623 0.9134535  1.13472422 0.98776324 0.86608698 0.94221182
 1.1414163  1.07862

(10, 12000)
t = 85 | collab = [1.13217832 0.94492639 1.1439157  0.99825433 0.85467504 0.97443238
 1.06072859 1.06708776 1.11551416 0.97730315]
Truth = [1.00471435 0.98809024 1.01432707 0.99687348 0.99279411 1.00887163
 1.00859588 0.99363476 1.00015696 0.97757315]
------------------------------
(10, 12000)
t = 86 | collab = [1.13137635 0.9511406  1.13909082 0.99958997 0.88020672 0.97605594
 1.06606719 1.057198   1.11523706 0.97141323]
Truth = [1.00471435 0.98809024 1.01432707 0.99687348 0.99279411 1.00887163
 1.00859588 0.99363476 1.00015696 0.97757315]
------------------------------
(10, 12000)
t = 87 | collab = [1.12652696 0.96338526 1.13867948 0.98896714 0.87464447 0.98074048
 1.06950754 1.06171382 1.11139025 0.96618195]
Truth = [1.00471435 0.98809024 1.01432707 0.99687348 0.99279411 1.00887163
 1.00859588 0.99363476 1.00015696 0.97757315]
------------------------------
(10, 12000)
t = 88 | collab = [1.12474169 0.93678377 1.11798276 0.96178318 0.873261   0.97161398
 1.06920825 1.0576

(10, 12000)
t = 113 | collab = [1.12320737 0.89499607 1.07003108 0.95733107 0.87056184 0.98751972
 1.0881551  1.08222501 1.12558999 0.86785327]
Truth = [1.00471435 0.98809024 1.01432707 0.99687348 0.99279411 1.00887163
 1.00859588 0.99363476 1.00015696 0.97757315]
------------------------------
(10, 12000)
t = 114 | collab = [1.13162147 0.90202056 1.06022197 0.95120078 0.86426486 0.99117722
 1.08467281 1.08102567 1.11644577 0.87445346]
Truth = [1.00471435 0.98809024 1.01432707 0.99687348 0.99279411 1.00887163
 1.00859588 0.99363476 1.00015696 0.97757315]
------------------------------
(10, 12000)
t = 115 | collab = [1.13420594 0.9082923  1.04860661 0.94803593 0.87292511 0.97718949
 1.08718189 1.07548535 1.11735184 0.85634014]
Truth = [1.00471435 0.98809024 1.01432707 0.99687348 0.99279411 1.00887163
 1.00859588 0.99363476 1.00015696 0.97757315]
------------------------------
(10, 12000)
t = 116 | collab = [1.14018722 0.91021864 1.04772987 0.94928627 0.87806144 0.97901901
 1.07922128 1.

(10, 12000)
t = 141 | collab = [1.18311952 0.86247945 1.04334361 0.97648234 0.92385507 0.98048072
 1.0734494  1.12891847 1.04503864 0.93816809]
Truth = [1.00471435 0.98809024 1.01432707 0.99687348 0.99279411 1.00887163
 1.00859588 0.99363476 1.00015696 0.97757315]
------------------------------
(10, 12000)
t = 142 | collab = [1.17857015 0.85663328 1.05780957 0.97658862 0.93292686 0.9869438
 1.06936987 1.11692826 1.04069306 0.92660813]
Truth = [1.00471435 0.98809024 1.01432707 0.99687348 0.99279411 1.00887163
 1.00859588 0.99363476 1.00015696 0.97757315]
------------------------------
(10, 12000)
t = 143 | collab = [1.17676409 0.85415858 1.05866189 0.96957921 0.93571594 0.98288795
 1.06469859 1.12088728 1.04390104 0.9251899 ]
Truth = [1.00471435 0.98809024 1.01432707 0.99687348 0.99279411 1.00887163
 1.00859588 0.99363476 1.00015696 0.97757315]
------------------------------
(10, 12000)
t = 144 | collab = [1.16792794 0.85801524 1.07077657 0.97738116 0.92865801 0.97874741
 1.05532914 1.1

In [ ]:
np.random.seed(1234)

sim_list = tqdm(list(range(args_quadratic['sim_num'])))
results_quad = Parallel(n_jobs=-1)(delayed(run_sim)(args_quadratic, i) for i in sim_list)

In [ ]:
def sim_data(args):
    # Simulate true beta
    hist_lk_list = [[[]] for n in range(args['N'])]
    diff_hist_lk_list = [[[]] for n in range(args['N'])]
    
    m_counter = jnp.zeros(args['N'], dtype=int)
    k_counter = jnp.ones(args['N'], dtype=int)
    
    key = jax.random.key(args['seed'])
    # Sample parameter for each site
    if args['d']== 1:
        mu1_true = jax.random.normal(key=key, shape=(args['N'], )) * np.sqrt(args['Sigma_true']) + args['mu_true']
    else:
        mu1_true = jax.random.multivariate_normal(key=key, mean=args['mu_true'].flatten(), cov=args['Sigma_true'], shape=(args['N'], ))

    for t in range(args['T']):
        for n in range(args['N']):
            # Reassign the variable
            m = m_counter[n]
            k = k_counter[n]
            beta = mu1_true[n]

            # Compute inner product of beta
            if args['d'] > 1:
                beta_term = 0
                for degree in range(args['d']):
                    beta_term += beta[degree] * ((k)**(degree)) * (args['delta']**(degree+1))
            else:
                beta_term = beta * args['delta']

            # Accumulate lk
            lkm1 = args['l_0'] if k == 1 else hist_lk_list[n][m][-1][1]
            lk = lkm1 + beta_term + jax.random.normal(key=key, shape=(1, )) * args['sigma0']

            # Append to the list
            hist_lk_list[n][m].append((k.item(), lk.item()))
            diff_hist_lk_list[n][m].append((k.item(), (lk-lkm1).item()))

            if lk > args['r_limit'] and t != args['T']-1:
                m_counter = m_counter.at[n].add(1)
                k_counter = k_counter.at[n].set(0)
            
                hist_lk_list[n].append([])
                diff_hist_lk_list[n].append([])
                
        # Increment k
        k_counter += 1
        
    return hist_lk_list, diff_hist_lk_list, mu1_true

In [ ]:
def centralized_gibbs_sampler(args, diff_lk_input):
    # Set seed
    key = jax.random.key(args['seed'])
    
    # Set parameters 
    N = args['N']; d = args['d'];
    mu_mu = args['mu_mu']; Sigma_mu = args['Sigma_mu'];
    nu_Sigma = args['nu_Sigma']; Sigma_Sigma = args['Sigma_Sigma'];
    counter = 0
    
    # Set parameters we want to learn
    mu = jnp.ones((d, 1)); beta = jnp.ones((d, N)); Sigma = jnp.eye(d);
    
    # Inverse
    Sigma_mu_inv = jnp.linalg.inv(Sigma_mu) if d>1 else jnp.reciprocal(Sigma_mu)
    Sigma_Sigma_inv = jnp.linalg.inv(Sigma_Sigma) if d>1 else jnp.reciprocal(Sigma_Sigma)
    Sigma_inv = jnp.linalg.inv(Sigma) if d>1 else jnp.reciprocal(Sigma_Sigma)
    
    # Initialize empty list
    length = (args['gibbs_T'] - args['warm_up']) // args['space']
    mu_sequence = jnp.zeros((d, length))
    beta_sequence = jnp.zeros((d, N, length))
    Sigma_sequence = jnp.zeros((d, d, length))
    
    # Start running Gibbs sampling
    for t in range(args['gibbs_T']):
        # Sample mu
        sum_beta = jnp.sum(beta, axis=1).reshape(-1, 1)
        Sigma_tilde_inv = Sigma_mu_inv + N * Sigma_inv
        Sigma_tilde = jnp.linalg.inv(Sigma_tilde_inv) if d>1 else jnp.reciprocal(Sigma_tilde_inv)
        b_tilde = jnp.dot(Sigma_mu_inv, mu_mu) + jnp.dot(Sigma_inv, sum_beta)
        if d>1:
            new_mu = jax.random.multivariate_normal(key=key, mean=jnp.dot(Sigma_tilde, b_tilde).flatten(), cov=Sigma_tilde).reshape(-1, 1)
        else:
            new_mu = jax.random.normal(key=key) * np.sqrt(Sigma_tilde) + np.dot(Sigma_tilde, b_tilde)
        
        # Sample Sigma
        cent_mu = new_mu - beta
        if d>1:
            new_Sigma = scipy.stats.invwishart.rvs(df=(nu_Sigma + N).item(), scale=Sigma_Sigma + cent_mu @ cent_mu.T, random_state=args['seed'])
        else:
            new_Sigma = scipy.stats.invgamma.rvs(a=(nu_Sigma + N).item()/2, scale=(Sigma_Sigma+ cent_mu @ cent_mu.T)/2, random_state=args['seed'])

        # Sample beta_i's
        new_beta = jnp.zeros((d, N))
        new_Sigma_inv = jnp.linalg.inv(new_Sigma) if d>1 else jnp.reciprocal(new_Sigma)
        for n in range(N):
            A_T = jnp.copy(new_Sigma_inv)
            b_T = jnp.dot(new_Sigma_inv, new_mu)
            for k, diff_lk in diff_lk_input[n]:
                tt_k = jnp.multiply(args['delta'] ** np.arange(1, d+1), k ** np.arange(d)).reshape(-1, 1) if d>1 else args['delta']
                tt_k_matrix = jnp.outer(tt_k, tt_k) if d>1 else args['delta']**2
                
                A_T += tt_k_matrix / (args['sigma0']**2)
                b_T += diff_lk * tt_k / (args['sigma0']**2)
            
            A_T_inv = np.linalg.inv(A_T) if d>1 else 1/A_T
            if d>1:
                new_beta = new_beta.at[:, n].set(jax.random.multivariate_normal(key=key, mean=jnp.dot(A_T_inv, b_T).flatten(), cov=A_T_inv))
            else:
                new_beta = new_beta.at[:, n].set(jax.random.normal(key=key)*np.sqrt(A_T_inv) + A_T_inv*b_T)
        
        # Complete one gibbs sampling 
        old_mu = mu; old_Sigma = Sigma; old_beta = beta;
        mu = new_mu; Sigma = new_Sigma; beta = new_beta;
        print(beta)
        # Append to the list after warm-up period
        if t > args['warm_up'] and t % args['space'] == 0:
            mu_sequence = mu_sequence.at[:, counter].set(mu.flatten() if d>1 else mu) 
            beta_sequence = beta_sequence.at[:, :, counter].set(beta)
            Sigma_sequence = Sigma_sequence.at[:, :, counter].set(Sigma)
            counter += 1
    
#     beta_est = np.mean(beta_sequence, axis=-1)
# #     beta_var = 
    
    return mu_sequence, beta_sequence, Sigma_sequence

In [ ]:
args_linear = {
        # Hierarchical params
        'delta': 1, 
        'mu_mu': jnp.array([5]), 'Sigma_mu': jnp.array([5]), # Prior of mu
        'nu_Sigma': jnp.array([4]) , 'Sigma_Sigma': jnp.array([2]), # Prior of Sigma
        'sigma0': jnp.array([0.5]), 'l_0': jnp.array([0]), 
        'mu_true': jnp.array([1]), 'Sigma_true': jnp.array([0.1]),
        # Simulation params
        'N': 10, 'M': 20, 'K': 80, 'S': 20, 'r_limit': 10, 'd': 1, 'T': 100, 'seed': 1234,
        # Sampling parameters
        'gibbs_T': 5000,  'warm_up': 2500, 'space': 2
    }
args_quadratic = {
        # Hierarchical params
        'delta': 1, 
        'mu_mu': jnp.array([[5], [1]]), 'Sigma_mu': jnp.array([[5, 0], [0, 5]]), # Prior of mu
        'nu_Sigma': jnp.array([4]) , 'Sigma_Sigma': jnp.array([[5, 0], [0, 1]]), # Prior of Sigma
        'sigma0': jnp.array([0.5]), 'l_0': jnp.array([0]), 
        'mu_true': jnp.array([[1], [0.3]]), 'Sigma_true': jnp.array([[0.01, 0], [0, 0.001]]),
        # Simulation params
        'N': 10, 'M': 20, 'K': 80, 'S': 20, 'r_limit': 40, 'd': 2, 'T': 100, 'seed': 1234,
        # Sampling parameters
        'gibbs_T': 5000, 'warm_up': 2500, 'space': 2
    }
curr_args = copy.deepcopy(args_linear)
hist_lk, hist_diff_lk, mu_true = sim_data(curr_args)
hist_diff_lk_flat = []
for n in range(curr_args['N']):
    lk_list = list(itertools.chain.from_iterable(hist_diff_lk[n]))
    hist_diff_lk_flat.append(lk_list)

start_time = time.time()
mu_list, beta_list, Sigma_list =  centralized_gibbs_sampler(curr_args, hist_diff_lk_flat)
print("--- %s seconds ---" % (time.time() - start_time))

In [ ]:
for k, diff_lk in diff_lk_input[n]:
    tt_k = np.multiply(args['delta'] ** np.arange(1, d+1), k ** np.arange(d)).reshape(-1, 1) if d>1 else args['delta']
    tt_k_matrix = tt_k @ tt_k.T if d>1 else args['delta']**2

    noise = np.random.laplace(scale=noise_sigma) if noise_sigma>0 else 0
    diff_lk += noise

    A[n*d:(n+1)*d, n*d:(n+1)*d] += tt_k_matrix / (args['sigma0']**2)
    b[n*d:(n+1)*d] += diff_lk * tt_k / (args['sigma0']**2)